# 中文新聞主題多分類 — 微調 BERT (Colab)

用預訓練中文模型 `hfl/chinese-roberta-wwm-ext` 微調做 6 類新聞主題分類,
與 TF-IDF baseline、word2vec+BiLSTM 對比。**記得開 GPU 執行階段。**

## 1. 安裝套件並 clone repo

In [ ]:
!pip -q install transformers
!git clone https://github.com/Upisofsht/Nlp2026.git
%cd Nlp2026/final_project
# PyTorch 已內建於 Colab;資料集 data/news_dataset.csv 隨 repo 一併下載。

## 2. 載入資料 + 標籤編碼 + 切分

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '.')
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/news_dataset.csv')
df['text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')
labels = sorted(df['category'].unique())
label2idx = {l: i for i, l in enumerate(labels)}
df['y'] = df['category'].map(label2idx)
tr, te = train_test_split(df, test_size=0.2, stratify=df['y'], random_state=42)
print('train', len(tr), 'test', len(te), 'classes', len(labels))

## 3. 載入 tokenizer 並建立 Dataset

In [ ]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = 'hfl/chinese-roberta-wwm-ext'
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True,
                             padding='max_length', max_length=MAX_LEN)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item

train_ds = NewsDataset(tr['text'], tr['y'])
test_ds = NewsDataset(te['text'], te['y'])

## 4. 微調 BERT(約幾分鐘)

In [ ]:
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels))

args = TrainingArguments(
    output_dir='bert_out',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    logging_steps=20,
    report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

## 5. 評估 + 混淆矩陣

In [ ]:
from src.evaluate import evaluate_predictions, plot_confusion_matrix

pred_logits = trainer.predict(test_ds).predictions
pred_idx = pred_logits.argmax(axis=1)
pred = [labels[i] for i in pred_idx]
true = [labels[i] for i in te['y']]
m = evaluate_predictions(true, pred)
print('BERT:', f"acc={m['accuracy']:.4f} macroF1={m['macro_f1']:.4f}")
print(m['report'])
plot_confusion_matrix(true, pred, labels, out_path='confusion_bert.png')